# E1 Distillation Baseline
Author: Arush Arora

## Introduction
This notebook explores the development of a GREP-PRISM Classical Planning LLM. The workflow for the GREP-PRISM model includes the following steps:
1. Tokenize a classical planning prompt that includes a relevant scene graph in text.
2. Obtain embeddings from a trained R-PEARL model that produces Graph Positional Encodings (GREPs) in $\mathbb{R}^d$ and add them to select word embeddings from the prompt semantically representing nodes in the scene graph.
3. Feed the prompt, containing a mix of Fourier and graphically positioned word embeddings, to the distilled Llama3.2:0.5b PRISM model that will process the Classical Planning prompt without the scene graph to return the next action of the robot.

_Note_: The training loop will remove the final softmax layer of the transformer for Cross-Entropy Loss evaluation.

## Libraries

In [1]:
from torch_geometric.utils import to_scipy_sparse_matrix
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import torch

def clear_gpu_memory():
    # 1. Delete the model/trainer variables if they exist in global scope
    # globals() checks ensure we don't error if the variable isn't defined
    if 'model' in globals(): del globals()['model']
    if 'trainer' in globals(): del globals()['trainer']

    # 2. Run Garbage Collection to release Python references
    gc.collect()

    # 3. Clear PyTorch's CUDA cache
    torch.cuda.empty_cache()

    # 4. Optional: Verify
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Run this before your training block
clear_gpu_memory()

GPU Memory Allocated: 0.00 GB


In [3]:
import json
from ast import literal_eval
import string
import re

In [4]:
import gc
import torch

device = 'cuda'

def find_cuda_tensors():
    tensors = []
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) or (hasattr(obj, 'data') and torch.is_tensor(obj.data)):
                # Check if the tensor is on a CUDA device
                if obj.is_cuda:
                    tensors.append(obj)
        except Exception:
            pass # ignore errors from objects that are not tensors or do not have is_cuda attribute

    return tensors

## The R-PEARL GNN

The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $\utilde{A}$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $P(\cdot) = I(\cdot)$, where $I$ is the identity function):
$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$
$$\Phi(\utilde{X}, \utilde{S}, \mathcal{H}) = \utilde{X}^{(L)}$$
$$\utilde{X}^{(0)} = \utilde{X} \qquad \utilde{X}^{(l)} = P\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \utilde{S}^k\utilde{X}^{(l - 1)}{\utilde{H}}_k^{(l)}\Bigg)\Bigg]$$

In [5]:
import torch

from torch import nn
from torch.utils.checkpoint import checkpoint
from torch_geometric.nn import TAGConv
from torch_geometric.data import Data

We add a device for management-purposes.

In [6]:
class GCN(nn.Module):
    """
    A simple TAG-based graph convolutional backbone that returns node embeddings.

    Args:
        in_channels (int): Number of input features per node
        hidden_channels (int): Number of hidden features per node
        num_layers (int): Number of convolution layers (must be >= 2)
        skip_connection (bool): Whether to use skip connections
        use_batch_norm (bool): Whether to use batch normalization
        k (int): Order of TAGConv polynomial (K)

    Returns:
        torch.Tensor: Node embeddings of shape [num_nodes, hidden_channels]
    """

    def __init__(self,
        in_channels,
        hidden_channels,
        num_layers,
        skip_connection=False,
        use_batch_norm=False,
        dropout=0.5,
        k: int = 3,
    ):
        super().__init__()
        if num_layers < 2:
            raise ValueError("GCN requires at least 2 layers.")

        self.convs = nn.ModuleList()
        self.k = k
        self.convs.append(TAGConv(in_channels, hidden_channels, K=self.k))
        self.norms = nn.ModuleList()
        for _ in range(num_layers - 2):
            self.convs.append(TAGConv(hidden_channels, hidden_channels, K=self.k))
            self.norms.append(nn.LayerNorm(hidden_channels))
            if use_batch_norm:
                self.norms.append(nn.BatchNorm1d(hidden_channels))
        self.convs.append(TAGConv(hidden_channels, hidden_channels, K=self.k))
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(p=dropout)
        self.skip_connection = skip_connection
        self.embedding_dim = hidden_channels

    def forward(self, data: Data):
        """
        Forward pass through the GCN.

        Args:
            data (Data): PyTorch Geometric Data object containing node features (x)
                        and edge indices (edge_index)

        Returns:
            torch.Tensor: Output node embeddings [num_nodes, hidden_channels]
        """
        try:
            device = next(self.parameters()).device
        except StopIteration:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        data.x = data.x.to(device)
        data.edge_index = data.edge_index.to(device)
        x0, edge_index = data.x, data.edge_index
        x_prev = x0
        x = x0.clone()
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x_prev, edge_index)
            if i < len(self.norms):
                x = self.norms[i](x)
            x = self.relu(x)
            x = self.dropout(x)
            if self.skip_connection and i > 0:
                x = x + x_prev
            x_prev = x
        x = self.convs[-1](x, edge_index)
        return x

### Random Graph Positional Encodings (R-PEARL)
The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$\utilde{Q} \in \mathbb{R}^{M \times N} \qquad \utilde{Q} \sim \mathcal{N}(0, \utilde{I}) \qquad \utilde{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\utilde{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\utilde{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\utilde{P}$:$$\utilde{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \utilde{S}^k\mathbf{q}^{(m)} {\utilde{H}}_k\bigg)$$$$\utilde{P} = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \utilde{P}^{(m)}$$

In [7]:
class RandomGNNPositionalEncodings(nn.Module):
    """
    Random graph positional encodings (R-PEARL).

    Args:
        pe_hidden_channels (int): Hidden dimension for the GCN
        pe_num_layers (int): Number of layers in the GCN
        d_model (int): Output dimension
        num_samples (int): Number of random samples (M) to use
        dropout (float): Dropout rate of the GCN associated.
        k (int): Convolution depth of the GCN.
        use_layer_norm (bool): Whether to use layer normalization
    """

    def __init__(self,
        pe_hidden_channels,
        pe_num_layers,
        d_model,
        num_samples=30,
        dropout=0.1,
        k: int = 3,
        use_layer_norm=True,
    ):
        super().__init__()
        # Create a GCN that takes 1-dimensional random features
        self.pe_gcn = GCN(
            1, pe_hidden_channels, pe_num_layers, skip_connection=True, dropout=dropout, k=k
        )
        # Add a final projection to ensure output is d_model dimensions
        self.output_projection = nn.Linear(pe_hidden_channels, d_model)
        self.dropout = nn.Dropout(dropout)
        self.use_layer_norm = use_layer_norm
        if self.use_layer_norm:
            self.layer_norm = nn.LayerNorm(d_model)
        else:
            self.batch_norm = nn.BatchNorm1d(d_model)
        self.M = num_samples

    def forward(self, data):
        # Move input data to the model's device.
        try:
            device = next(self.parameters()).device
        except StopIteration:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        data.x = data.x.to(device)
        data.edge_index = data.edge_index.to(device)
        X, edge_index = data.x, data.edge_index

        # Generate random node embeddings for positional encoding
        num_nodes = X.shape[0]
        Q = torch.randn((num_nodes, self.M), device=device)

        # Process random embeddings individually through GCN
        P_m = []

        for i in range(self.M):

            def _pe_block(q_col, edge_idx, _dummy):
                q_data = Data(x=q_col.unsqueeze(-1), edge_index=edge_idx)
                pe_local = self.pe_gcn(q_data)
                pe_local = self.dropout(pe_local)
                pe_local = self.output_projection(pe_local)
                return pe_local

            dummy = Q.new_ones(1, requires_grad=True, device=device)
            pe = checkpoint(_pe_block, Q[:, i], edge_index, dummy, use_reentrant=False)
            P_m.append(pe)
        # checkpoint

        P = torch.stack(P_m, dim=-1)
        pooled_pe = P.mean(dim=-1)
        if self.use_layer_norm:
            pooled_pe = self.layer_norm(pooled_pe)
        else:
            pooled_pe = self.batch_norm(pooled_pe)
        return pooled_pe

## Transformer

$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$The Transformer architecture follows that of the Llama3.2-3B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $\utilde{E}$ and $\utilde{\tilde{X}}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } \utilde{E} = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } \utilde{X} = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\utilde{X} = \utilde{\tilde{X}} + \utilde{P}$$

$${\utilde{Z}}_{1:t}^{(L)} = \operatorname{Trf}\bigg({\utilde{X}}_{1:t}, {\mathcal{T}}_l\bigg) \qquad {\mathcal{T}}_l = \begin{bmatrix}
{\utilde{Q}}_l & {\utilde{K}}_l & {\utilde{V}}_l & \left({\utilde{W}}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big({\utilde{Z}}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(\utilde{E}, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

In [8]:
from huggingface_hub import whoami

whoami()

{'type': 'user',
 'id': '6935e7c3462d178c8443d18c',
 'name': 'arar1234',
 'fullname': 'A A',
 'isPro': False,
 'avatarUrl': '/avatars/d6b9b5a20a47363d566920adb7b9338f.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'Alelab (Download)',
   'role': 'fineGrained',
   'createdAt': '2025-12-31T04:18:25.626Z',
   'fineGrained': {'canReadGatedRepos': True,
    'global': [],
    'scoped': [{'entity': {'_id': '6935e7c3462d178c8443d18c',
       'type': 'user',
       'name': 'arar1234'},
      'permissions': ['inference.serverless.write']}]}}}}

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

models = {
    'qwen': ('Qwen/Qwen2.5-0.5B-Instruct', 896),
    'llama': ('meta-llama/Llama-3.2-3B-Instruct', 3072)
}

code = 'llama'

name, emb_dim = models[code]
model = AutoModelForCausalLM.from_pretrained(name)
tokenizer = AutoTokenizer.from_pretrained(name)
tokenizer.pad_token = tokenizer.eos_token

model, tokenizer

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

(LlamaForCausalLM(
   (model): LlamaModel(
     (embed_tokens): Embedding(128256, 3072)
     (layers): ModuleList(
       (0-27): 28 x LlamaDecoderLayer(
         (self_attn): LlamaAttention(
           (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
           (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
           (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
           (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
         )
         (mlp): LlamaMLP(
           (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
           (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
           (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
           (act_fn): SiLUActivation()
         )
         (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
         (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
       )
     )
     (norm): LlamaRMSNorm((30

From `modular_qwen2.py`'s forward pass:
```python
@deprecate_kwarg("past_key_value", new_name="past_key_values", version="4.58")
def forward(
    self,
    hidden_states: torch.Tensor,
    position_embeddings: tuple[torch.Tensor, torch.Tensor],
    attention_mask: Optional[torch.Tensor],
    past_key_values: Optional[Cache] = None,
    cache_position: Optional[torch.LongTensor] = None,
    **kwargs: Unpack[FlashAttentionKwargs],
) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
    ...
```

## Data Processing

After defining the internal mechanisms of the GREP-PRISM architecture employed in this project, we turn to the processing of all Classical Planning prompts in `gpt_gen_formatted.json` using the scene-graph decoding algorithms included in this work's parent project, PRISM.

### Data Loading

We now have all architectures needed to run the full training algorithm. We begin by importing all Classical Planning prompt data with the scene graphs attached.

In [10]:
with open("data/eval/gpt_gen_formatted.json", "r") as f:
    raw = json.load(f)
# Take the first element of the first conversation to check out that scene graph.
first_prompt = raw[0]['conversations'][0]['content']
display(first_prompt[:100])

scene_graph_text = re.findall(r"Scene graph:(.*)", first_prompt)[0]
display(scene_graph_text[:100])

graph_dict = literal_eval(scene_graph_text)  # handles the single quotes safely
display(graph_dict.keys())

"task: I need a shovel. Is there one in the scene?Scene graph:{'objects': [{'name': 'house_1', 'coord"

"{'objects': [{'name': 'house_1', 'coords': [-1, -1]}, {'name': 'house_2', 'coords': [-3, -1]}, {'nam"

dict_keys(['objects', 'regions', 'object_connections', 'region_connections', 'robot_location'])

In [11]:
from trl.trainer import SFTTrainer, SFTConfig
from datasets import load_dataset
from prism import scene_graph_parser

train_dataset = load_dataset("json", data_files=["data/eval/gpt_gen_formatted.json"], split="train")


def _add_messages(example):
    example["messages"] = example["conversations"]
    return example


def _tokenize_with_conversations(example):
    tokenized = tokenizer.apply_chat_template(
        example["messages"], tokenize=True, return_dict=True
    )
    tokenized["conversations"] = example["conversations"]
    tokenized["messages"] = example["messages"]
    return tokenized


train_dataset = train_dataset.map(_add_messages)
train_dataset = train_dataset.map(_tokenize_with_conversations)
#  train_dataset = train_dataset.map(scene_graph_parser._parse_scene_graph_dictionary_from_conversation)

train_dataset

Dataset({
    features: ['conversations', 'messages', 'input_ids', 'attention_mask'],
    num_rows: 990
})

### Accomodating Erroneous Graphs

We see that SPINE's `parse_graph` function may not be enough to process eroneous graphs. We thus define a function that converts a JSON graph into an `nx.Graph` object from the package `networkx`. We configure the `safe_parse_graph` function to handle graphs that may be incorrectly defined (missing nodes in the edge list, etc.).

In [12]:
from typing import Dict, Optional, Tuple
from scipy.spatial.transform import Rotation
import networkx as nx
import numpy as np
from copy import deepcopy
from spine.mapping.graph_util import parse_graph_coord

def safe_parse_graph(
    data: Dict[str, Dict[str, str]],
    custom_data: Optional[Dict[str, Dict[str, str]]] = {},
    rotation: Optional[Rotation] = None,
    utm_origin: Optional[np.ndarray] = None,
    flip_coords=False,
) -> Tuple[nx.Graph, str]:
    """Parse scene graph in `data` into a networkx object.

    Parameters
    ----------
    data : Dict[str, Dict[str, str]]
        graph where keys-values are nodes-attributes
    rotation : Optional[Rotation]
        current rotation of robot

    Returns
    -------
    Tuple[nx.Graph, str]
        Networkx and string of json
    """
    origin = np.array([0, 0])
    data = deepcopy(data)  # don't modify input data
    as_str = str(data)

    if utm_origin is not None:
        origin = utm_origin

    if len(custom_data):
        add_keys = ["regions", "region_connections", "objects", "object_connections"]
        for key in add_keys:
            if key in data and key in custom_data:
                data[key].extend(custom_data[key])

    G = nx.Graph()
    for node in data["objects"]:
        coords = parse_graph_coord(node["coords"], origin=origin, rotation=rotation)
        if flip_coords:
            raise ValueError()
            # print("flipping coords")
            coords = [coords[0], -coords[1]]

        node.pop("coords")
        name = node.pop("name")
        G.add_node(name, coords=coords, type="object", **node)

    for node in data["regions"]:
        assert "coords" in node, node
        c = node["coords"]
        # print(f"node: {node}, coords: {c}")
        coords = parse_graph_coord(node["coords"], origin=origin, rotation=rotation)

        if flip_coords:
            raise ValueError
            # print("flipping coords")
            coords = [coords[0], -coords[1]]
        node.pop("coords")
        name = node.pop("name")
        G.add_node(name, coords=coords, type="object", **node)

    for edge in data["object_connections"]:
        c1 = G.nodes[edge[0]]["coords"]
        c2 = G.nodes[edge[1]]["coords"]
        # print(f"edge: {edge}, c1, c2: {c1}, {c2}")
        dist = np.linalg.norm(np.array(c1) - np.array(c2))
        G.add_edge(edge[0], edge[1], type="object", weight=dist)

    for edge in data["region_connections"]:
        c1 = G.nodes[edge[0]]["coords"]
        c2 = G.nodes[edge[1]]["coords"]
        # print(f"edge: {edge}, c1, c2: {c1}, {c2}")
        dist = np.linalg.norm(np.array(c1) - np.array(c2))
        G.add_edge(edge[0], edge[1], type="region", weight=dist)

    return G, as_str

## Token-based Bucketize with test-case
This section implements bucketize assuming we're working with lists of token IDs, with an illustrative example. The helper function for finding subsets is implemented with some test cases.

In [13]:
CONVERSATION = [
    {
        "role": "user",
        "content": (
            "task: I need a shovel. Is there one near the shed in the scene?\n"
            "Scene graph:{"
            "'objects': ["
            "{'name': 'house_1', 'coords': [-1, -1]}, "
            "{'name': 'grocery_store_1', 'coords': [-5, -1]}, "
            "{'name': 'shed_1', 'coords': [1, 3]}], "
            "'regions': [{'name': 'field_11', 'coords': [0, 1]}], "
            "'object_connections': [['shed_1', 'field_11']], "
            "'region_connections': [], "
            "'robot_location': 'field_11'}"
        ),
    },
    {
        "role": "assistant",
        "content": (
            '{"primary_goal": "find a shovel near the shed", '
            '"relevant_graph": "field_11, shed_1, unobserved_node(shovel)", '
            '"reasoning": "The graph has one shed, shed_1, connected to '
            "field_11. There are two sheds total but only shed_1 is "
            "observed. I will explore field_11 to look for a shovel near "
            'shed_1.", '
            '"plan": "[goto(field_11), map_region(field_11)]"}'
        ),
    },
]

NODE_LIST = ["house_1", "grocery_store_1", "shed_1", "field_11"]# They appear 1,1,6, 8 times respectively.

MODEL_NAME = models[code][0]


### IMPORTED FIXTURES FROM THE TEST FILE
def tokenizer():
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    tok.pad_token = tok.eos_token
    return tok

def input_ids(tokenizer):
    prompt = tokenizer.apply_chat_template(
        CONVERSATION, tokenize=False, add_generation_prompt=False,
    )
    return tokenizer(prompt, return_tensors="pt")["input_ids"]
### END FIXTURES

tokenizer = tokenizer()
input_ids = input_ids(tokenizer)

In [14]:
#TODO turn into test case
def has_match(input_ids_b: list[int], to_match:list[int],start_pos:int):
    """ 
        For a single sequence, check if `to_match` is present at `start_pos`
    """
    end_pos = min(start_pos + len(to_match),len(input_ids_b))
    return input_ids_b[start_pos:end_pos] == to_match

display(has_match([1,2,3],[1,2],start_pos=0))
display(has_match([1,2,3],[1,2],start_pos=1))
display(has_match([1,2,3],[2,3],start_pos=0))
display(has_match([1,2,3],[2,3],start_pos=1))
display(has_match([1,2,3],[2,3],start_pos=3))

True

False

False

True

False

In [15]:
from collections import defaultdict


def bucketize_prompt(input_ids_b: list, node_token_seqs : list) -> defaultdict:
    """
    Helper function for associating full prompt words with their corresponding token indices.
    Uses parallel iteration through words alongside the token list.

    Args:
        input_ids (torch.Tensor): List of one-hot encodings for prompt.
        tokenizer (nn.Module): LLM tokenizer required to decode input IDs.

    Returns:
        bucket (dict): mappings for adding operation of positional encodings
            to respective tokens.
    """
    # Get map of words to token locations.
    buckets = defaultdict(set)
    for p_idx, p_token in enumerate(input_ids_b):
        for node_idx, node_token_seq in enumerate(node_token_seqs):
            if has_match(input_ids_b, to_match=node_token_seq,start_pos=p_idx):
                buckets[node_idx].add(p_idx)
    return buckets
print(f"Testing tokenize prompt on {input_ids.shape}")
buckets = bucketize_prompt(input_ids[0,:].tolist(), tokenizer.encode(NODE_LIST))    
buckets

Testing tokenize prompt on torch.Size([1, 261])


defaultdict(set, {})

## Test the new function on the whole dataset
Mind you: This decoding was generated by AI, but confirms the handrwitten code above is correct!

In [16]:
for ex_idx in range(min(5, len(train_dataset))):
    example = train_dataset[ex_idx]
    ids = example["input_ids"]
    decoded_prompt = tokenizer.decode(ids)

    # --- Parse node names from the scene-graph embedded in the prompt ---
    sg_match = re.search(r"[Ss]cene graph: ?(.*})", decoded_prompt)
    if sg_match is None:
        print(f"=== Example {ex_idx}: no scene graph found, skipping ===\n")
        continue
    scene_graph_dict = literal_eval(sg_match.group(1))
    try:
        nx_graph, _ = safe_parse_graph(scene_graph_dict)
    except KeyError as e:
        print(f"=== Example {ex_idx}: bad graph (missing node {e}), skipping ===\n")
        continue
    node_names = list(nx_graph.nodes)

    # --- Encode each node name into its token-ID sequence ---
    node_token_seqs = tokenizer.encode(node_names)  # list[list[int]]

    # --- Run bucketize_prompt ---
    buckets = bucketize_prompt(ids, node_token_seqs)

    # --- Print results ---
    print(f"{'='*80}")
    print(f"EXAMPLE {ex_idx}  ({len(ids)} tokens, {len(node_names)} nodes)")
    print(f"{'='*80}")

    # 1. Subset of the string prompt (first 300 chars).
    print(f"\n--- Prompt (first 300 chars) ---")
    print(decoded_prompt[:300], "..." if len(decoded_prompt) > 300 else "")

    # 2. Token IDs (first 40).
    print(f"\n--- Token IDs (first 40 of {len(ids)}) ---")
    print(ids[:40])

    # 3. Buckets: node_idx → set of start positions.
    print(f"\n--- Buckets (node_idx → match start positions) ---")
    for node_idx, starts in sorted(buckets.items()):
        print(f"  node {node_idx} ({node_names[node_idx]}): starts={sorted(starts)}")

    # 4. Decoded matches: for each bucket entry, decode the token range.
    print(f"\n--- Decoded matches ---")
    for node_idx, starts in sorted(buckets.items()):
        n_tokens = len(node_token_seqs[node_idx])
        name = node_names[node_idx]
        for s in sorted(starts):
            end = min(s + n_tokens, len(ids))
            matched_ids = ids[s:end]
            matched_text = tokenizer.decode(matched_ids)
            print(f"  {name:30s}  ids[{s}:{end}] = {matched_ids}  →  \"{matched_text}\"")

    print()

=== Example 0: bad graph (missing node 'shed_2'), skipping ===

EXAMPLE 1  (1439 tokens, 12 nodes)

--- Prompt (first 300 chars) ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 Feb 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

task: How many robots are currently observed, where are they located, and which is the southmost robot?scene graph: {'object ...

--- Token IDs (first 40 of 1439) ---
[128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 1419, 13806, 220, 2366, 21, 271, 128009, 128006, 882, 128007, 271, 8366, 25, 2650, 1690, 29807, 527, 5131, 13468, 11, 1405]

--- Buckets (node_idx → match start positions) ---

--- Decoded matches ---

EXAMPLE 2  (1011 tokens, 7 nodes)

--- Prompt (first 300 chars) ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 Feb 2026

<|eot_id|><|star

## Graph-Augmented LLM

The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$\renewcommand{\utilde}[1]{\underset{\sim}{#1}}$ $$\utilde{P} = \hat{\mathbb{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big)$$

$$\utilde{X} = \utilde{\tilde{X}} + \utilde{P}$$

$${\utilde{Z}}_{1:t}^{(L)} = \operatorname{Trf}\bigg({\utilde{X}}_{1:t}\bigg)$$

In [17]:
import re
from collections import defaultdict

import torch
from torch import nn


class GraphAugmentedLLM(nn.Module):
    """
    Graph-Augmented LLM (GREP-PRISM).

    Args:
        llm (nn.Module): LLM to perform classical planning.
        pe_model (nn.Module): R-PEARL positional-encodings model.
        tokenizer: (nn.Module): Tokenizer associated with LLM.
    """

    def __init__(self, llm: nn.Module, pe_model: nn.Module, tokenizer: nn.Module, pe_dim: int):
        super().__init__()
        self.llm = llm
        self.config = llm.config
        self.tokenizer = tokenizer

        # Place pe_model and pe_proj on the same device as the LLM so PEFT
        # wrapping (which only touches LoRA target modules) doesn't leave them on CPU.
        device = next(llm.parameters()).device
        self.pe_model = pe_model.to(device)
        self.pe_proj = nn.Linear(pe_dim, llm.config.hidden_size, device=device)

    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        graphs: list | None = None,
        **kwargs,
    ):
        # input_ids: [B,seq_len]
        
        
        # Let's start with LLM embeddings since we can compute them in batch.
        embeddings = (
            self.llm.get_input_embeddings()(input_ids)
                .squeeze(0)
                .to(input_ids.device)
        ) # [B,seq_len,d]


        # Now to inject positional encodings.
        for b in range(input_ids.shape[0]):
            graph = graphs[b]

            node_token_seqs = self.tokenizer.encode(graph.node_names)
            bucket = self.bucketize_prompt(input_ids[b,:].tolist(), node_token_seqs)
    
            # Get positional encodings.
            pe = self.pe_model(graph) # [n,d]
    
            for node_idx, match_idxes in bucket.items():
                for start in match_idxes:
                    max_len = input_ids.shape[1] #TO DO: worst case we'll inject PEs into some padded tokens but would be rare.
                    end = min(match_start_idx + len(node_token_seqs[node_idx]),max_len)
                    embeddings[b,start:end] = embeddings[b,start:end,:] + pe[node_idx,:]

        return self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs,
        )

    @classmethod
    def bucketize_prompt(cls, input_ids_b: list, node_token_seqs : list) -> defaultdict:
        """
        Helper function for associating full prompt words with their corresponding token indices.
        Uses parallel iteration through words alongside the token list.

        Args:
            input_ids (torch.Tensor): List of one-hot encodings for prompt.
            tokenizer (nn.Module): LLM tokenizer required to decode input IDs.

        Returns:
            bucket (dict): mappings for adding operation of positional encodings
                to respective tokens.
        """
        # Get map of words to token locations.
        buckets = defaultdict(set)
        for p_idx, p_token in enumerate(input_ids_b):
            for node_idx, node_token_seq in enumerate(node_token_seqs):
                if has_match(input_ids_b, to_match=node_token_seq,start_pos=p_idx):
                    buckets[node_idx].add(p_idx)
        return buckets

### Creating the Data Collator

We now engineer the penultimate class to begin the training sequence. The `DataCollatorForGraphAgumentedLLM` implementation below filters scene graphs from text-based Classical Planning prompts, safely parses them into `nx.Graph` objects using the `networkx` library in Python, and sanitizes them to be fit to the Task Planning scenario. It then batches the data in preparation for GREP-PRISM training.

In [18]:
from transformers.data.data_collator import DataCollatorForLanguageModeling
import torch
import torch_geometric.utils as pyg_utils


class DataCollatorForGraphAugmentedLLM(DataCollatorForLanguageModeling):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def __call__(self, features, return_tensors: Optional[str] = None):
        """Attach parsed PyG graphs for each conversation example."""
        messages = []
        pyg_graphs = []
        conversations = []
        offset_mappings = []
        sanitized_examples = []

        for example in features:
            pattern = r"[Ss]cene graph:"
            # Graph is in the first message
            prompt = self.tokenizer.decode(example['input_ids'])
            if re.search(pattern=pattern, string=prompt):
                scene_graph_text = re.findall(pattern + r" ?(.*})", prompt)[0]
                scene_graph_dict = literal_eval(scene_graph_text)  # handles the single quotes safely
            else:
                raise ValueError(f"No scene graph found in prompt: {prompt}")

            try:
                nx_graph, _ = safe_parse_graph(scene_graph_dict)
                node_names = list(nx_graph.nodes)
                coords = torch.tensor(
                    [nx_graph.nodes[node]["coords"] for node in node_names],
                    dtype=torch.float32,
                )

                pyg_graph = pyg_utils.from_networkx(nx_graph)
                pyg_graph.coords = coords
                pyg_graph.x = torch.zeros((coords.size(0), 1), dtype=torch.float32)
                pyg_graph.edge_index = pyg_graph.edge_index
                pyg_graph.node_names = node_names
                pyg_graph.node_types = [nx_graph.nodes[node]["type"] for node in node_names]
                pyg_graph.robot_location = scene_graph_dict.get("robot_location")
                pyg_graph.raw_scene_graph = scene_graph_dict
                pyg_graphs.append(pyg_graph)

                # Sanitize input IDs and attention masks.
                pattern = r"'object_connections':"
                decoded = self.tokenizer.decode(example["input_ids"])
                cleaned = re.sub(pattern + r' ?.*,', '', decoded)
                encoded = self.tokenizer(
                    cleaned, return_offsets_mapping=True, return_tensors="pt"
                )
                example['input_ids'] = encoded['input_ids'].squeeze().tolist()
                example['attention_mask'] = encoded['attention_mask'].squeeze().tolist()
                offset_mappings.append(encoded['offset_mapping'].squeeze().tolist())

                # Sanitize conversations and messages for later reinstallation.
                """
                if 'conversations' in example.keys() and messages in example.keys():
                    conv, mes = example['conversations'], example['messages']
                    conv[0]['content'] = re.sub(pattern + r' ?.*,', '', conv[0]['content'])
                    mes[0]['content'] = re.sub(pattern + r' ?.*,', '', mes[0]['content'])
                    conversations.append(conv)
                    messages.append(mes)
                """

                sanitized_examples.append(
                    {
                        k: v
                        for k, v in example.items()
                        if k not in {"conversations", "scene_graph", "messages"}
                    }
                )
            except Exception as e:
                print(f"Error parsing scene graph: {e}")
        # Call the parent collator to get the tensors (on sanitized examples so that it doesn't try to tensorize the scene graph/text)
        batch = super().__call__(sanitized_examples)
        batch["graphs"] = pyg_graphs
        batch['offset_mappings'] = offset_mappings
        if conversations and messages:
            batch['conversations'] = conversations
            batch['messages'] = messages
        return batch

In [19]:
collator = DataCollatorForGraphAugmentedLLM(tokenizer=tokenizer, mlm=False)
data = collator(train_dataset)

Error parsing scene graph: 'shed_2'


In [20]:
"""Testing"""

pe_model = RandomGNNPositionalEncodings(
    pe_hidden_channels=256, pe_num_layers=3, d_model=emb_dim, num_samples=40, dropout=0.1
).to(device)

graph_augmented_model = GraphAugmentedLLM(model, pe_model, tokenizer, emb_dim).to(device)

prompt = ("<|im_start|>system\nYou are a robotic agent who must complete tasks in a world.<|im_end|>\n"
          "<|im_start|>user\nGo to office_building_1 and using example_truck_1. Explicitly commend on "
          "the plan you will use to complete this task, and list your steps in bulletized fashion.<|im_end|>\n"
          "<|im_start|>assistant\n")

graph = data['graphs'][0]
graph

Data(
  edge_index=[2, 18],
  coords=[12, 2],
  type=[12],
  edge_type=[18],
  weight=[18],
  num_nodes=12,
  x=[12, 1],
  node_names=[12],
  node_types=[12],
  robot_location='example_road_1',
  raw_scene_graph={
    objects=[6],
    regions=[6],
    object_connections=[6],
    region_connections=[3],
    robot_location='example_road_1',
  }
)

## Experimental Section #1
This section experiments with the R-PEARL model, $\Phi\Big(\mathbf{q}^{(m)}, \utilde{S}, \mathcal{H}\Big)$, to investigate the spectral clustering properties of the filter banks of the model's PE GNN.
The premise of the spectral clustering experiment is as follows:
1. For an R-PEARL, we have the following equation:
$$\utilde{S} = \utilde{V} \Lambda \utilde{V}^H \qquad {\utilde{X}}^{(l)} = \sigma\bigg[\sum_{k = 0}^{K} {\utilde{S}}^k {\utilde{X}}^{(l - 1)} {\utilde{H}}^{(l)}_{k}\bigg]$$
2. We know from ESE 5140: "Graph Neural Networks" with Professor Alejandro Ribeiro that the frequency response for a MIMO GNN is defined as such:
$$\tilde{\utilde{H}}^{(l)}(\lambda) = \sum_{k = 0}^k {\utilde{H}}^{(l)}_{k} \lambda^k \in \mathbb{R}^{G_l \times F_l}$$
3. Thus, we assemble a spectral "sensitivity" diagonal matrix for output feature $f$ that stores the vector modulus of each column of the Frequency Response $\tilde{\utilde{H}}^{(l)}(\lambda_i)$ of the $k$ Graph Filter Banks ${\utilde{H}^{(l)}_k}$ for each eigenvalue $\lambda_i$ of the Graph Shift Operator $\utilde{S} = \utilde{V} \operatorname{diag}(\mathbf{\lambda}) \utilde{V}^H$. Therefore, each element of the resulting ${\utilde{Q}}^{(l)}_{f}$ matrix essentially stores the magnitude of Column $f$ of the convolution of Graph Filter Banks for Layer $l$ with $\utilde{S}$'s $i^{\text{th}}$ eigenvalue $\lambda_i$ in the frequency domain.
$${\utilde{Q}}^{(l)}_{f} = \operatorname{diag}\Bigg(\bigg\Vert \tilde{\utilde{H}}^{(l)}(\lambda_i) \cdot \mathbf{e}_f \bigg\Vert^2\Bigg)$$
4. After obtaining the spectral sensitivity matrix ${\utilde{Q}}^{(l)}_{f}$ for output feature $f$ and Layer $l$, we can finally compose the nodal spectral-clustering coloring vector $c^{(l)}_f \in \mathbb{R}^n$ for Hermitian GSOs $\utilde{S}$:
  $$\mathbf{c}^{(l)}_f = \Big(\utilde{V} \odot \utilde{V}^*\Big) \ \utilde{Q}^{(l)}_f \ \mathbf{1}$$
  We seek to self-multiply the modal (eigenvector) matrix of the Graph Shift Operator by itself using the Hadammard product with its complex conjugate to ensure that all values are positive, real numbers while maintaining monotonicity and thus relative ordering of element magnitudes (through respective moduli). Thus, we can multiply the result by the spectral sensitivities and sum by row to retrieve the vector of relative color values for visual clustering operations.

In [36]:
model = RandomGNNPositionalEncodings(pe_hidden_channels=256, pe_num_layers=5, d_model=emb_dim, num_samples=40).to(device)
model_state_dict = torch.load('output/training/gnn_weights.pt')
print(model_state_dict['pe_model'].keys())
model.load_state_dict(model_state_dict['pe_model'])

odict_keys(['pe_gcn.convs.0.bias', 'pe_gcn.convs.0.lins.0.weight', 'pe_gcn.convs.0.lins.1.weight', 'pe_gcn.convs.0.lins.2.weight', 'pe_gcn.convs.0.lins.3.weight', 'pe_gcn.convs.1.bias', 'pe_gcn.convs.1.lins.0.weight', 'pe_gcn.convs.1.lins.1.weight', 'pe_gcn.convs.1.lins.2.weight', 'pe_gcn.convs.1.lins.3.weight', 'pe_gcn.convs.2.bias', 'pe_gcn.convs.2.lins.0.weight', 'pe_gcn.convs.2.lins.1.weight', 'pe_gcn.convs.2.lins.2.weight', 'pe_gcn.convs.2.lins.3.weight', 'pe_gcn.convs.3.bias', 'pe_gcn.convs.3.lins.0.weight', 'pe_gcn.convs.3.lins.1.weight', 'pe_gcn.convs.3.lins.2.weight', 'pe_gcn.convs.3.lins.3.weight', 'pe_gcn.convs.4.bias', 'pe_gcn.convs.4.lins.0.weight', 'pe_gcn.convs.4.lins.1.weight', 'pe_gcn.convs.4.lins.2.weight', 'pe_gcn.convs.4.lins.3.weight', 'pe_gcn.norms.0.weight', 'pe_gcn.norms.0.bias', 'pe_gcn.norms.1.weight', 'pe_gcn.norms.1.bias', 'pe_gcn.norms.2.weight', 'pe_gcn.norms.2.bias', 'output_projection.weight', 'output_projection.bias', 'layer_norm.weight', 'layer_norm.bi

<All keys matched successfully>

In [22]:
model.pe_gcn.convs[0].lins[0]

Linear(1, 256, bias=False)

In [23]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

class MplColorHelper:
    def __init__(self, cmap_name, min_val, max_val):
        self.cmap_name = cmap_name
        self.cmap = plt.get_cmap(cmap_name)
        self.norm = mcolors.Normalize(vmin=min_val, vmax=max_val)
        self.scalarMap = plt.cm.ScalarMappable(norm=self.norm, cmap=self.cmap)

    def get_rgb_str(self, val):
        # Returns an RGBA tuple, convert to an HTML color string if needed by pyvis
        return mcolors.to_hex(self.scalarMap.to_rgba(val))

In [33]:
%matplotlib inline

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from random import randint
from pyvis.network import Network
from IPython.display import display, HTML
from torch_geometric.utils import to_dense_adj, to_networkx

index = randint(0, len(train_dataset) - 1)
graph = data['graphs'][index]

adj = to_dense_adj(graph.edge_index, max_num_nodes=graph.num_nodes)[0]
eigenvalues, eigenvectors = torch.linalg.eigh(adj)
eigenvalues, eigenvectors = eigenvalues.to(device), eigenvectors.to(device)

def get_freq_response(layer: int, eigenvalues: torch.Tensor):
    graph_filter = model.pe_gcn.convs[layer].lins
    F_in, F_out = graph_filter[0].weight.shape[1], graph_filter[0].weight.shape[0]
    n = eigenvalues.shape[0]
    result = torch.zeros(n, F_in, F_out, device=eigenvalues.device)
    for k, lin in enumerate(graph_filter):
        H_lk = lin.weight.T
        for i in range(n):
            result[i] += (eigenvalues[i] ** k) * H_lk
    return result

freq = get_freq_response(4, eigenvalues)
sensitivity = torch.linalg.norm(freq, dim=1) ** 2
clusters = (eigenvectors * eigenvectors.conj()) @ sensitivity
clusters

tensor([[ 0.3022,  0.3603,  0.3451,  ...,  0.3516,  0.3207,  0.3524],
        [ 4.6389,  4.6402,  5.3089,  ...,  5.2284,  5.1608,  4.9169],
        [15.4917, 15.3171, 17.7189,  ..., 17.5095, 17.2584, 16.2946],
        ...,
        [11.1551, 11.0372, 12.7551,  ..., 12.6328, 12.4183, 11.7301],
        [ 7.1092,  7.0999,  8.1342,  ...,  7.9508,  7.9267,  7.5512],
        [ 2.7726,  2.8200,  3.1704,  ...,  3.0741,  3.0866,  2.9867]],
       device='cuda:0', grad_fn=<MmBackward0>)

In [34]:
import networkx as nx
from pyvis.network import Network

graphX = to_networkx(graph, node_attrs=['node_names'], edge_attrs=None, to_undirected=True)
mapping = {i: name for i, name in enumerate(graph.node_names)}
graphX = nx.relabel_nodes(graphX, mapping)

node_values = clusters[:, 0].squeeze().tolist()
node_colors_helper = MplColorHelper("RdYlBu", min(node_values), max(node_values))
for i, name in enumerate(graph.node_names):
    color_hex = node_colors_helper.get_rgb_str(node_values[i])
    graphX.nodes[name]['color'] = color_hex
    graphX.nodes[name]['value'] = node_values[i]
    graphX.nodes[name]['title'] = f"Value: {node_values[i]}"
    graphX.nodes[name]['label'] = str(name)

graphX.nodes()

NodeView(('trees_1', 'courtyard_1', 'courtyard_2', 'courtyard_3', 'courtyard_6', 'courtyard_4', 'courtyard_5', 'sidewalk_1', 'sidewalk_2', 'parking_lot_1'))

In [35]:
net = Network(notebook=True, height="500px", width="100%", bgcolor="#222222", font_color="white", cdn_resources='in_line')
net.from_nx(graphX)
net.save_graph("interactive_graph.html")

## Experimental Section #2
This section makes use of the dual graph, $G^*$, to train a parallel R-PEARL to produce edge-embeddings for the GREP-PRISM framework as well (in hopes of removing the text-based edge list).

The steps to obtain the Graph Shift Operator (GSO) for $G^*$, $S^*$, are as follows:
1. Obtain $\sqrt{\mathbf{w}} = \big[\sqrt{w_1}, \cdots, \sqrt{w_m}\big]^\top$.
2. Construct $B$, where $\utilde{S} = \utilde{L} = \utilde{B}\utilde{W}\utilde{B}^\top$ and $\utilde{W} = \operatorname{diag}(\mathbf{w})$

In [ ]:
# Initialize variables.
n, m = graph.num_nodes, graph.edge_index.size(1)
exp_device = graph.edge_index.device

# Prepare sparse incidence matrix by distilling indices and values.
edge_ids = torch.arange(m, device=exp_device)
row_indices = torch.cat([graph.edge_index[0], graph.edge_index[1]])
col_indices = torch.cat([edge_ids, edge_ids])
values = torch.cat([
    torch.tensor(-1., device=exp_device).expand(m),
    torch.tensor(1., device=exp_device).expand(m),
])

# Construct sparse (n, m) incidence matrix.
B = torch.sparse_coo_tensor(
    indices=torch.stack([row_indices, col_indices]),
    values=values,
    size=(n, m),
).coalesce()

# Get sparse unweighted Dual GSO matrix.
dual = (B.T @ B).coalesce()

# If weights are present, prepare final weighted values.
if graph.weight is not None:
    # Square-root weights for quadratic linear-algebraic equation.
    w_sqrt = torch.sqrt(torch.abs(graph.weight) + 1e-12)
    values, indices = dual.values(), dual.indices()
    weighted_values = values * w_sqrt[indices[0]] * w_sqrt[indices[1]]
else:
    weighted_values = dual.values()

S_star = torch.sparse_coo_tensor(
    indices=dual.indices(),
    values=weighted_values,
    size=dual.size(),
).coalesce()

# Construct Dual-Graph Data object.
dual_graph = Data(
    edge_index=S_star.indices(),
    coords=graph.coords,
    type=graph.edge_type,
    edge_type=graph.type,
    edge_weight=S_star.values(),

    num_nodes=m
)

dual_graph

In [ ]:
pe_model = pe_model.to(device)
pe_model(graph)

In [ ]:
emb = model.get_input_embeddings()
tokenized = tokenizer(prompt, return_offsets_mapping=True, return_tensors='pt')
tokenized.input_ids = tokenized.input_ids.to(device)
tokenized.offset_mapping = tokenized.offset_mapping.squeeze().tolist()
embeddings = emb(tokenized.input_ids)
embeddings = embeddings.squeeze().to(device)
embeddings.shape

In [ ]:
embeddings

In [ ]:
# Test forward pass with new bucketize_prompt implementation
output = graph_augmented_model(
    input_ids=tokenized.input_ids, 
    attention_mask=tokenized.attention_mask, 
    graphs=[graph]
)
output

In [ ]:
pe = graph_augmented_model.pe_model(graph)
pos_enc = defaultdict(lambda: torch.Tensor(size=pe.size(), device=pe.device))
for i, word in enumerate(graph.node_names):
    pos_enc[word] = pe[i]
pos_enc

In [ ]:
new_embeddings = embeddings.clone()

bucket = bucketize_prompt(tokenized.input_ids.tolist(), tokenizer.encode(graph.node_names))

for word, token in bucket.items():
    if word in pos_enc:
        for pos in token:
            new_embeddings[pos] = new_embeddings[pos] + pos_enc[word]

(new_embeddings * embeddings).sum()

In [ ]:
output = graph_augmented_model(
    input_ids=tokenized.input_ids,
    num_return_sequences=1,
    attention_mask=tokenized.attention_mask,
    pad_token_id=tokenizer.eos_token_id,
    graphs=[graph]
)

# output_text = output
output_text = torch.argmax(output['logits'], dim=-1)

tokenizer.decode(output_text.squeeze(), skip_special_tokens=True)

In [ ]:
prompt = """
System: you are a robotic agent who must complete tasks in a world.
User: Go to office_building_1 and using example_truck_1. Explicitly commend on the plan you will use to complete this task, and list your steps in bulletized fashion.
Assistant: """

prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids
ids = torch.tensor(train_dataset['input_ids'][0], dtype=torch.int)
bucket = GraphAugmentedLLM.bucketize_prompt(prompt_ids, tokenizer, train_dataset['graphs'][0].node_names)

In [ ]:
# Quick SFT sanity check without graph augmentation
baseline_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

sft_baseline_config = SFTConfig(
    output_dir='/home/arushar/source_code/GREP-PRISM/output/training',
    max_steps=100,
    max_length=None,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    logging_steps=1,
    logging_strategy="steps",
    save_strategy="no",
    #evaluation_strategy="no",
    report_to="none",
    dataloader_pin_memory=False,
)

train_subset = train_dataset.select(range(8))

'''
baseline_trainer = SFTTrainer(
    model=model,
    args=sft_baseline_config,
    train_dataset=train_subset,
    processing_class=tokenizer,
    data_collator=baseline_collator,
)

# baseline_train_result = baseline_trainer.train()
# baseline_train_result
'''

In [ ]:
pe_model = RandomGNNPositionalEncodings(
    pe_hidden_channels=256, pe_num_layers=3, d_model=emb_dim, num_samples=40, dropout=0.1, # 0.05 0.01
    use_layer_norm=True
).to(device)

graph_augmented_model = GraphAugmentedLLM(model, pe_model, tokenizer, emb_dim).to(device)
graph_augmented_model.llm.requires_grad_(False)

collator = DataCollatorForGraphAugmentedLLM(tokenizer=tokenizer, mlm=False)

baseline_trainer = SFTTrainer(
    model=graph_augmented_model,
    args=sft_baseline_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=collator,
)

baseline_train_result = baseline_trainer.train()
baseline_train_result